# Module 13: Production Deployment Patterns

**Snowpark ML Fundamentals — Week 3 | Lunch & Learn (Instructor Demo)**

---

## Learning Objectives
- Call registered models from SQL
- Build a scoring stored procedure
- Schedule batch inference with Snowflake Tasks
- Monitor prediction distributions

## Key Concept
> Once a model is in the **Model Registry**, it becomes a first-class Snowflake
> object. Any user with the right role can call it from SQL, stored procedures,
> or scheduled tasks — **no Python runtime required** for consumers.

## Prerequisites
This notebook assumes `CHURN_FS_MODEL` exists from notebook 12.

---

In [1]:
%load_ext autoreload
%autoreload 2

## 13.1 Setup

In [2]:
import sys
sys.path.insert(0, '../src')

import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.registry.model_registry import (
    get_registry,
    list_versions,
    get_model_metrics,
)

session = create_session(env_path='../.env')

current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

# Verify model exists
versions_df = list_versions(registry, 'CHURN_FS_MODEL')
print("CHURN_FS_MODEL versions:")
print(versions_df[['name', 'aliases']].to_string())

CHURN_FS_MODEL versions:
  name                                  aliases
0   V1  ["PRODUCTION","DEFAULT","FIRST","LAST"]


## 13.2 SQL Inference

Registered models are callable as **SQL functions**. This means any analyst,
BI tool, or data pipeline can score data without Python.

The syntax is: `MODEL_NAME!PREDICT(<columns...>)`

In [3]:
# SQL inference using the registered model
# The model is called as a row-level function: MODEL!PREDICT(*)
sql_predictions = session.sql(f"""
    WITH features AS (
        SELECT
            DAYS_SINCE_LAST_ORDER, ORDERS_30D, ORDERS_90D, ORDERS_365D,
            ORDERS_TOTAL, SPEND_30D, SPEND_90D, SPEND_365D, SPEND_TOTAL,
            AVG_ORDER_VALUE, TOTAL_ITEMS, AVG_ITEMS_PER_ORDER,
            TOTAL_PAGE_VIEWS, TOTAL_CLICKS, TOTAL_SUPPORT_TICKETS,
            TOTAL_EMAIL_ENGAGEMENTS, INTERACTIONS_30D, SUPPORT_TICKETS_30D,
            INTERACTIONS_90D, DAYS_SINCE_LAST_INTERACTION, AVG_DURATION_SECONDS
        FROM {current_db}.{current_schema}.CUSTOMER_RFM_FEATURES r
        JOIN {current_db}.{current_schema}.CUSTOMER_BEHAVIORAL_FEATURES b
            ON r.CUSTOMER_ID = b.CUSTOMER_ID
        LIMIT 10
    )
    SELECT *, {current_db}.{current_schema}.CHURN_FS_MODEL!PREDICT(*) AS PREDICTION
    FROM features
""")

print("SQL-based inference results:")
sql_predictions.show()

SQL-based inference results:
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"DAYS_SINCE_LAST_ORDER"  |"ORDERS_30D"  |"ORDERS_90D"  |"ORDERS_365D"  |"ORDERS_TOTAL"  |"SPEND_30D"  |"SPEND_90D"  |"SPEND_365D"  |"SPEND_TOTAL"  |"AVG_ORDER_VALUE"  |"TOTAL_ITEMS"  |"AVG_ITEMS_PER_ORDER"  |"TOTAL_PAGE_VIEWS"  |"TOTAL_CLICKS"  |"TOTAL_SUPPORT_TICKETS"  |"TOTAL_EMAIL_ENGAGEMENTS"  |"INTERACTIONS_30D"  |"SUPPORT_TICKETS_30D"  |"INTERACTIONS_90D"  |"DAYS_SINCE_LAST_INTERACTION"  |"AVG_DURATION_SECONDS"  |"PREDICTION"             |
---------------------------------------------------------------------

## 13.3 Stored Procedure for Batch Scoring

A stored procedure encapsulates the full scoring workflow:
1. Load the production model by alias
2. Retrieve features from the Feature Store
3. Score all customers
4. Write predictions to an output table

In [ ]:
# Create a stored procedure for batch scoring
session.sql(f"""
    CREATE OR REPLACE PROCEDURE {current_db}.{current_schema}.SCORE_CHURN_CUSTOMERS()
    RETURNS STRING
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.10'
    PACKAGES = ('snowflake-snowpark-python', 'snowflake-ml-python')
    HANDLER = 'run'
    AS
    $$
def run(session):
    from snowflake.ml.registry import Registry

    db = session.get_current_database().replace('"', '')
    schema = session.get_current_schema().replace('"', '')

    # Load production model by alias
    registry = Registry(session=session, database_name=db, schema_name=schema)
    model_ref = registry.get_model('CHURN_FS_MODEL')
    production = model_ref.version('production')

    # Join features using SQL — explicit column refs avoid ambiguity
    # (both tables share _FEATURE_TIMESTAMP; SQL aliases keep it clean)
    features = session.sql(f'''
        SELECT r.CUSTOMER_ID,
               r.DAYS_SINCE_LAST_ORDER, r.ORDERS_30D, r.ORDERS_90D, r.ORDERS_365D,
               r.ORDERS_TOTAL, r.SPEND_30D, r.SPEND_90D, r.SPEND_365D, r.SPEND_TOTAL,
               r.AVG_ORDER_VALUE, r.TOTAL_ITEMS, r.AVG_ITEMS_PER_ORDER,
               b.TOTAL_PAGE_VIEWS, b.TOTAL_CLICKS, b.TOTAL_SUPPORT_TICKETS,
               b.TOTAL_EMAIL_ENGAGEMENTS, b.INTERACTIONS_30D, b.SUPPORT_TICKETS_30D,
               b.INTERACTIONS_90D, b.DAYS_SINCE_LAST_INTERACTION, b.AVG_DURATION_SECONDS
        FROM {{db}}.{{schema}}.CUSTOMER_RFM_FEATURES r
        JOIN {{db}}.{{schema}}.CUSTOMER_BEHAVIORAL_FEATURES b
            ON r.CUSTOMER_ID = b.CUSTOMER_ID
    ''')

    feature_cols = [
        'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D', 'ORDERS_365D',
        'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_365D', 'SPEND_TOTAL',
        'AVG_ORDER_VALUE', 'TOTAL_ITEMS', 'AVG_ITEMS_PER_ORDER',
        'TOTAL_PAGE_VIEWS', 'TOTAL_CLICKS', 'TOTAL_SUPPORT_TICKETS',
        'TOTAL_EMAIL_ENGAGEMENTS', 'INTERACTIONS_30D', 'SUPPORT_TICKETS_30D',
        'INTERACTIONS_90D', 'DAYS_SINCE_LAST_INTERACTION', 'AVG_DURATION_SECONDS',
    ]

    # Score
    predictions = production.run(features.select(['CUSTOMER_ID'] + feature_cols),
                                function_name='predict')

    # Write to output table
    output_table = f'{{db}}.{{schema}}.CHURN_PREDICTIONS'
    predictions.write.mode('overwrite').save_as_table(output_table)

    # Count from materialized table (avoids re-running model inference)
    count = session.table(output_table).count()
    return f'Scored {{count}} customers'
    $$
""").collect()

print("Stored procedure created: SCORE_CHURN_CUSTOMERS()")

Stored procedure created: SCORE_CHURN_CUSTOMERS()


In [5]:
# Execute the stored procedure
result = session.sql(f"CALL {current_db}.{current_schema}.SCORE_CHURN_CUSTOMERS()").collect()
print("Result:", result[0][0])

# Check output table
predictions_table = session.table(f"{current_db}.{current_schema}.CHURN_PREDICTIONS")
print(f"\nPredictions table: {predictions_table.count()} rows")
predictions_table.show(5)

Result: Scored 2000 customers

Predictions table: 2000 rows
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"DAYS_SINCE_LAST_ORDER"  |"ORDERS_30D"  |"ORDERS_90D"  |"ORDERS_365D"  |"ORDERS_TOTAL"  |"SPEND_30D"  |"SPEND_90D"  |"SPEND_365D"  |"SPEND_TOTAL"  |"AVG_ORDER_VALUE"  |"TOTAL_ITEMS"  |"AVG_ITEMS_PER_ORDER"  |"TOTAL_PAGE_VIEWS"  |"TOTAL_CLICKS"  |"TOTAL_SUPPORT_TICKETS"  |"TOTAL_EMAIL_ENGAGEMENTS"  |"INTERACTIONS_30D"  |"SUPPORT_TICKETS_30D"  |"INTERACTIONS_90D"  |"DAYS_SINCE_LAST_INTERACTION"  |"AVG_DURATION_SECONDS"  |"output_feature_0"  |
----------------

## 13.4 Snowflake Tasks: Scheduled Scoring

A **Snowflake Task** can schedule the stored procedure to run on a cron.
This is the simplest way to deploy batch inference without external orchestration.

> **Note**: The task is created in SUSPENDED state. In production, you would
> `ALTER TASK ... RESUME` to activate it.

In [6]:
# Create a scheduled task (SUSPENDED — not activated)
session.sql(f"""
    CREATE OR REPLACE TASK {current_db}.{current_schema}.CHURN_SCORING_TASK
        WAREHOUSE = '{session.get_current_warehouse().replace('"', '')}'
        SCHEDULE = 'USING CRON 0 6 * * * America/New_York'
        COMMENT = 'Daily churn scoring at 6 AM ET'
    AS
        CALL {current_db}.{current_schema}.SCORE_CHURN_CUSTOMERS()
""").collect()

print("Task created: CHURN_SCORING_TASK (SUSPENDED)")
print("Schedule: Daily at 6:00 AM ET")
print("")
print("To activate in production:")
print(f"  ALTER TASK {current_db}.{current_schema}.CHURN_SCORING_TASK RESUME;")

Task created: CHURN_SCORING_TASK (SUSPENDED)
Schedule: Daily at 6:00 AM ET

To activate in production:
  ALTER TASK MLDS_Q.PREDICTIONS.CHURN_SCORING_TASK RESUME;


## 13.5 Monitoring Patterns

Production models need monitoring. Key signals:
1. **Prediction distribution** — has it shifted from baseline?
2. **Feature drift** — are input features changing?
3. **Model metrics** — compare registered metrics with live performance

In [7]:
# Query prediction distribution from the output table
distribution = session.sql(f"""
    SELECT
        "output_feature_0" AS PREDICTION,
        COUNT(*) AS COUNT,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS PERCENTAGE
    FROM {current_db}.{current_schema}.CHURN_PREDICTIONS
    GROUP BY "output_feature_0"
    ORDER BY "output_feature_0"
""")

print("Prediction distribution:")
distribution.show()

Prediction distribution:
-----------------------------------------
|"PREDICTION"  |"COUNT"  |"PERCENTAGE"  |
-----------------------------------------
|0             |1459     |72.95         |
|1             |541      |27.05         |
-----------------------------------------



In [8]:
# Compare with registered training metrics
training_metrics = get_model_metrics(registry, 'CHURN_FS_MODEL', 'V1')
print("Registered training metrics:")
for metric, value in training_metrics.items():
    print(f"  {metric}: {value}")

print("\nMonitoring checklist:")
print("  - Is the churn prediction rate consistent with training data?")
print("  - Are feature distributions stable? (check CUSTOMER_RFM_FEATURES)")
print("  - Set alerts if prediction rate shifts > 10% from baseline.")

Registered training metrics:
  accuracy: 0.860789
  precision: 0.8421052631578947
  recall: 0.6956521739130435
  f1_score: 0.761904761904762

Monitoring checklist:
  - Is the churn prediction rate consistent with training data?
  - Are feature distributions stable? (check CUSTOMER_RFM_FEATURES)
  - Set alerts if prediction rate shifts > 10% from baseline.


## 13.6 Architecture Summary

The complete production architecture for ML in Snowflake:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                     SNOWFLAKE ML ARCHITECTURE                          │
│                                                                        │
│  ┌──────────┐     ┌────────────────┐     ┌──────────────┐              │
│  │ Raw Data │────►│ dbt / Feature   │────►│ Feature      │              │
│  │ (Tables) │     │ Engineering    │     │ Store        │              │
│  └──────────┘     └────────────────┘     └──────┬───────┘              │
│                                                  │                      │
│                           ┌──────────────────────┼──────────────┐       │
│                           │                      │              │       │
│                           ▼                      ▼              │       │
│                   ┌──────────────┐     ┌────────────────┐       │       │
│                   │ Training Set │────►│ Model Training │       │       │
│                   │ (spine join) │     │ (XGBoost etc.) │       │       │
│                   └──────────────┘     └───────┬────────┘       │       │
│                                                │                │       │
│                                                ▼                │       │
│                                        ┌──────────────┐         │       │
│                                        │   Model      │         │       │
│                                        │   Registry   │         │       │
│                                        │  (aliases,   │         │       │
│                                        │   metrics)   │         │       │
│                                        └──────┬───────┘         │       │
│                                               │                 │       │
│                           ┌───────────────────┼─────────┐       │       │
│                           │                   │         │       │       │
│                           ▼                   ▼         ▼       │       │
│                   ┌──────────────┐  ┌──────────┐ ┌──────────┐  │       │
│                   │ Stored Proc  │  │ SQL      │ │ Python   │  │       │
│                   │ + Task       │  │ Inference│ │ Inference│  │       │
│                   │ (scheduled)  │  │ (ad-hoc) │ │ (app)    │  │       │
│                   └──────┬───────┘  └──────────┘ └──────────┘  │       │
│                          │                                      │       │
│                          ▼                                      │       │
│                   ┌──────────────┐     ┌────────────────┐       │       │
│                   │ Predictions  │────►│ Monitoring     │◄──────┘       │
│                   │ Table        │     │ (drift, dist)  │               │
│                   └──────────────┘     └────────────────┘               │
│                                                                        │
│  Note: Snowpark Container Services (SPCS) can host real-time           │
│  endpoints for low-latency inference. Requires Enterprise edition.     │
└─────────────────────────────────────────────────────────────────────────┘
```

### What we covered in Week 3:

| Notebook | Topic | Production Pattern |
|----------|-------|-------------------|
| 10 | Model Versioning | Immutable versions with metrics comparison |
| 11 | Model Lifecycle | Alias-based promotion (zero-downtime) |
| 12 | FS → Registry | Training-serving consistency |
| 13 | Production Patterns | SQL inference, stored procedures, tasks, monitoring |

## Cleanup

Uncomment to clean up all Week 3 resources.

In [9]:
# from snowpark_fundamentals.registry.model_registry import delete_model

# # Delete models
# delete_model(registry, 'CHURN_PREDICTOR_W3')
# delete_model(registry, 'CHURN_FS_MODEL')
# print("Models deleted.")

# # Drop production artifacts
# session.sql(f"DROP TABLE IF EXISTS {current_db}.{current_schema}.CHURN_PREDICTIONS").collect()
# session.sql(f"DROP TASK IF EXISTS {current_db}.{current_schema}.CHURN_SCORING_TASK").collect()
# session.sql(f"DROP PROCEDURE IF EXISTS {current_db}.{current_schema}.SCORE_CHURN_CUSTOMERS()").collect()
# print("Production artifacts dropped.")

## Key Takeaways

1. Registered models are **SQL-callable** — `MODEL!PREDICT(...)` works anywhere
2. **Stored procedures** encapsulate the full scoring workflow
3. **Snowflake Tasks** schedule batch inference without external orchestration
4. **Monitoring** prediction distributions catches drift before it impacts business
5. The complete loop: Data → Features → Training → Registry → Scoring → Monitoring

---

**End of Week 3** — You now have production-grade patterns for the full Snowflake ML lifecycle.

In [10]:
session.close()